# Stage 02 — Extração de características PSD

Objetivo: transformar cada época filtrada em um vetor de potência espectral. Com 128 canais e duas bandas, cada época terá 256 características.


## 1. Importações


In [118]:
from pathlib import Path

import numpy as np
from scipy.signal import periodogram


## 2. Caminhos e parâmetros


In [119]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_01_load_and_epoching"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_02_feature_extraction_psd"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLING_RATE = 256

BANDS = [(4, 8), (8, 13), (13, 30), (30, 45)]

print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)


Entrada: /home/jobson/Documentos/DSP_classifier/DSP-classifier/processed_data/stage_01_load_and_epoching
Saída: /home/jobson/Documentos/DSP_classifier/DSP-classifier/processed_data/stage_02_feature_extraction_psd


## 3. Descobrir os arquivos produzidos pela Stage 20


In [120]:
data_files = sorted(INPUT_DIR.glob("*_inner_epochs.npy"))
print(f"Sessões encontradas: {len(data_files)}")
print("Exemplo:", data_files[0].name)


Sessões encontradas: 30
Exemplo: sub-01_ses-01_inner_epochs.npy


## 4. Carregar e inspecionar uma sessão

O eixo 0 são épocas, o eixo 1 são canais, o eixo 2 são amostras.


In [121]:
example_data = np.load(data_files[0])
print("Formato:", example_data.shape)
print("Tipo:", example_data.dtype)
print("Todos os valores são finitos:", np.isfinite(example_data).all())


Formato: (80, 128, 640)
Tipo: float64
Todos os valores são finitos: True


## 5. Função de extração

O periodograma produz uma potência para cada frequência. Para cada banda, aplicamos uma máscara e somamos a potência ao longo das frequências.


In [122]:
def extract_psd_features(data):
    frequencies, psd = periodogram(
        data,
        fs=SAMPLING_RATE,
        axis=-1
    )

    total_power = psd.sum(axis = -1)

    features_by_band = []

    for low_frequency, high_frequency in BANDS:
        frequency_mask = (
            (frequencies >= low_frequency)
            & (frequencies < high_frequency)
        )

        band_power = psd[:, :, frequency_mask].sum(axis=-1)

        relative_power = band_power / total_power

        features_by_band.append(relative_power)

    return np.stack(features_by_band, axis=-1)

## 6. Testar a extração em uma sessão


In [123]:
example_features = extract_psd_features(example_data)
print("Entrada:", example_data.shape)
print("Saída (épocas, canais, características):", example_features.shape)
print("Mínimo e máximo:", example_features.min(), example_features.max())


Entrada: (80, 128, 640)
Saída (épocas, canais, características): (80, 128, 4)
Mínimo e máximo: 0.00027212159947128295 0.5777051140857739


## 7. Processar e salvar todas as sessões


In [124]:
for data_path in data_files:
    session_name = data_path.name.replace("_inner_epochs.npy", "")
    labels_path = INPUT_DIR / f"{session_name}_inner_epochs_labels.npy"

    data = np.load(data_path)
    labels = np.load(labels_path)
    features = extract_psd_features(data)

    if len(features) != len(labels):
        raise ValueError(f"Épocas e rótulos incompatíveis em {session_name}")

    np.save(OUTPUT_DIR / f"{session_name}_features_psd.npy", features)
    np.save(OUTPUT_DIR / f"{session_name}_labels.npy", labels)
    print(f"{session_name}: {features.shape}")

print("Stage 30 finalizado.")


sub-01_ses-01: (80, 128, 4)
sub-01_ses-02: (80, 128, 4)
sub-01_ses-03: (40, 128, 4)
sub-02_ses-01: (80, 128, 4)
sub-02_ses-02: (80, 128, 4)
sub-02_ses-03: (80, 128, 4)
sub-03_ses-01: (40, 128, 4)
sub-03_ses-02: (80, 128, 4)
sub-03_ses-03: (60, 128, 4)
sub-04_ses-01: (80, 128, 4)
sub-04_ses-02: (80, 128, 4)
sub-04_ses-03: (80, 128, 4)
sub-05_ses-01: (80, 128, 4)
sub-05_ses-02: (80, 128, 4)
sub-05_ses-03: (80, 128, 4)
sub-06_ses-01: (80, 128, 4)
sub-06_ses-02: (80, 128, 4)
sub-06_ses-03: (56, 128, 4)
sub-07_ses-01: (80, 128, 4)
sub-07_ses-02: (80, 128, 4)
sub-07_ses-03: (80, 128, 4)
sub-08_ses-01: (80, 128, 4)
sub-08_ses-02: (80, 128, 4)
sub-08_ses-03: (40, 128, 4)
sub-09_ses-01: (80, 128, 4)
sub-09_ses-02: (80, 128, 4)
sub-09_ses-03: (80, 128, 4)
sub-10_ses-01: (80, 128, 4)
sub-10_ses-02: (80, 128, 4)
sub-10_ses-03: (80, 128, 4)
Stage 30 finalizado.
